# Impacts of Random Under Sampling

### Classification without RUS and SMOTE - (SVM, and GRU)

In [25]:
import pickle
import numpy as np
from collections import Counter
from imblearn.under_sampling import EditedNearestNeighbours

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

# ── Load hybrid_72 only ──
BASE = "./final_split_data_HybridNorm_timesteps"
datasets = {}
folder = f"{BASE}/hybrid_72"
X_train, y_train = load_split(f"{folder}/train_set.pkl")
X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
X_test,  y_test  = load_split(f"{folder}/test_set.pkl")
datasets["hybrid_72"] = {
    "X_train": X_train, "y_train": y_train,
    "X_val":   X_val,   "y_val":   y_val,
    "X_test":  X_test,  "y_test":  y_test,
}

# ── Random Undersampling Function ──
def random_undersample(X, y, nsep_keep, seed=42):
    rng = np.random.default_rng(seed)
    idx_nsep = np.where(y == 0)[0]
    idx_sep  = np.where(y == 1)[0]
    if nsep_keep is None or nsep_keep >= len(idx_nsep):
        idx_nsep_sampled = idx_nsep
    else:
        idx_nsep_sampled = rng.choice(idx_nsep, size=nsep_keep, replace=False)
    idx_combined = np.concatenate([idx_nsep_sampled, idx_sep])
    idx_combined = rng.permutation(idx_combined)
    return X[idx_combined], y[idx_combined]

# ── Build Undersampled Variants (8000 and 4000 only) ──
undersample_configs = [
    ("nsep_8000", 8000),
    ("nsep_4000", 4000),
]

undersampled_datasets = {}
ds = datasets["hybrid_72"]
for cfg_name, nsep_keep in undersample_configs:
    variant_name = f"hybrid_72_{cfg_name}"
    X_res, y_res = random_undersample(ds["X_train"], ds["y_train"], nsep_keep)
    undersampled_datasets[variant_name] = {
        "X_train": X_res,        "y_train": y_res,
        "X_val":   ds["X_val"],  "y_val":   ds["y_val"],
        "X_test":  ds["X_test"], "y_test":  ds["y_test"],
    }

# ── Summary ──
print(f"{'Variant':<25} {'Train shape':<25} {'NSEP (0)':>10} {'SEP (1)':>10} {'Ratio':>8}")
print("─" * 85)
for name, d in undersampled_datasets.items():
    counts = np.bincount(d["y_train"].astype(int))
    ratio  = counts[0] / counts[1]
    print(f"{name:<25} {str(d['X_train'].shape):<25} {counts[0]:>10} {counts[1]:>10} {ratio:>7.1f}x")

# ══════════════════════════════════════════════════════════════
# ENN CLEANING — applied to each undersampled variant
# ══════════════════════════════════════════════════════════════
print("\nApplying ENN cleaning to undersampled variants …")

for variant_name in list(undersampled_datasets.keys()):
    d        = undersampled_datasets[variant_name]
    n, t, c  = d["X_train"].shape
    X_flat   = d["X_train"].reshape(n, -1)
    y_int    = d["y_train"].astype(int)

    enn = EditedNearestNeighbours(n_neighbors=3)
    X_clean_flat, y_clean = enn.fit_resample(X_flat, y_int)
    X_clean = X_clean_flat.reshape(-1, t, c)

    print(f"\n── {variant_name} ──")
    print(f"  Before : {d['X_train'].shape}  NSEP={(y_int==0).sum()}  SEP={(y_int==1).sum()}")
    print(f"  After  : {X_clean.shape}  NSEP={(y_clean==0).sum()}  SEP={(y_clean==1).sum()}")
    print(f"  Removed {n - X_clean.shape[0]} samples")

    undersampled_datasets[variant_name] = {
        "X_train": X_clean,          "y_train": y_clean.astype(np.float32),
        "X_val":   d["X_val"],       "y_val":   d["y_val"],
        "X_test":  d["X_test"],      "y_test":  d["y_test"],
    }

print(f"\n✅  undersampled_datasets : {list(undersampled_datasets.keys())}")

Variant                   Train shape                 NSEP (0)    SEP (1)    Ratio
─────────────────────────────────────────────────────────────────────────────────────
hybrid_72_nsep_8000       (8118, 72, 10)                  8000        118    67.8x
hybrid_72_nsep_4000       (4118, 72, 10)                  4000        118    33.9x

Applying ENN cleaning to undersampled variants …

── hybrid_72_nsep_8000 ──
  Before : (8118, 72, 10)  NSEP=8000  SEP=118
  After  : (7947, 72, 10)  NSEP=7829  SEP=118
  Removed 171 samples

── hybrid_72_nsep_4000 ──
  Before : (4118, 72, 10)  NSEP=4000  SEP=118
  After  : (3931, 72, 10)  NSEP=3813  SEP=118
  Removed 187 samples

✅  undersampled_datasets : ['hybrid_72_nsep_8000', 'hybrid_72_nsep_4000']


### Vectorization for SVM

In [32]:
# ══════════════════════════════════════════════════════════════
# CATCH22 EXTRACTION — hybrid_72 undersampled variants only
# ══════════════════════════════════════════════════════════════
import os
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22

SAVE_DIR = "./catch22_features_timesteps"
os.makedirs(SAVE_DIR, exist_ok=True)

def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()

def extract_catch22(X: np.ndarray, label: str = "", n_jobs: int = -1, chunk_size: int = 500) -> np.ndarray:
    n_samples, n_timepoints, n_channels = X.shape

    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️  {n_bad} non-finite values in {label} — replacing with per-channel median")
        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]
            col[~np.isfinite(col)] = np.nanmedian(col)
            X[:, :, c] = col

    X_sktime = X.transpose(0, 2, 1).astype(np.float64)
    chunks   = [X_sktime[i:i + chunk_size] for i in range(0, n_samples, chunk_size)]
    print(f"    {label} — {n_samples} samples in {len(chunks)} chunks …", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk) for chunk in chunks
    )

    X_feat = np.vstack(results)
    expected_cols = 22 * n_channels
    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected catch22 output shape: {X_feat.shape}  "
        f"(expected ({n_samples}, {expected_cols}))"
    )
    return X_feat

# ── Extract val and test once (shared across all 72 variants) ──
print("\nExtracting shared val and test for hybrid_72 …")
d_base = datasets["hybrid_72"]
X_va = extract_catch22(d_base["X_val"],  label="hybrid_72/val",  n_jobs=-1)
X_te = extract_catch22(d_base["X_test"], label="hybrid_72/test", n_jobs=-1)

# ── Extract train for each hybrid_72 variant ──────────────────
catch22_72 = {}

for variant_name, d in undersampled_datasets.items():
    if not variant_name.startswith("hybrid_72"):
        continue

    print(f"\n── {variant_name} ──────────────────────────────────────")
    X = d["X_train"]
    print(f"  Finite: {np.isfinite(X).all()}  Min: {X.min():.4f}  Max: {X.max():.4f}  NaNs: {(~np.isfinite(X)).sum()}")

    X_tr = extract_catch22(d["X_train"], label=f"{variant_name}/train", n_jobs=-1)

    catch22_72[variant_name] = {
        "X_train": X_tr, "y_train": d["y_train"],
        "X_val":   X_va, "y_val":   d_base["y_val"],
        "X_test":  X_te, "y_test":  d_base["y_test"],
    }

    print(f"  ✓  train {X_tr.shape}  val {X_va.shape}  test {X_te.shape}")

print("\n✓ All hybrid_72 variants vectorized and stored in catch22_72")


Extracting shared val and test for hybrid_72 …
    hybrid_72/val — 1780 samples in 4 chunks …
    hybrid_72/test — 3559 samples in 8 chunks …

── hybrid_72_nsep_8000 ──────────────────────────────────────
  Finite: True  Min: -5.8109  Max: 60.6786  NaNs: 0
    hybrid_72_nsep_8000/train — 7947 samples in 16 chunks …
  ✓  train (7947, 220)  val (1780, 220)  test (3559, 220)

── hybrid_72_nsep_4000 ──────────────────────────────────────
  Finite: True  Min: -5.5434  Max: 60.6786  NaNs: 0
    hybrid_72_nsep_4000/train — 3931 samples in 8 chunks …
  ✓  train (3931, 220)  val (1780, 220)  test (3559, 220)

✓ All hybrid_72 variants vectorized and stored in catch22_72


### Missing value imputation from Catch22 - Loading the data

In [33]:
# ══════════════════════════════════════════════════════════════
# IMPUTE — hybrid_72 undersampled variants (from catch22_72)
# ══════════════════════════════════════════════════════════════
from sklearn.impute import SimpleImputer
import numpy as np

catch22_72_imputed = {}

for variant_name, d in catch22_72.items():

    X_tr, y_tr = d["X_train"].copy(), d["y_train"]
    X_va, y_va = d["X_val"].copy(),   d["y_val"]
    X_te, y_te = d["X_test"].copy(),  d["y_test"]

    n_bad_tr = (~np.isfinite(X_tr)).sum()
    n_bad_va = (~np.isfinite(X_va)).sum()
    n_bad_te = (~np.isfinite(X_te)).sum()

    if n_bad_tr > 0 or n_bad_va > 0 or n_bad_te > 0:
        print(f"  ⚠️  [{variant_name}] non-finite values — "
              f"train: {n_bad_tr}  val: {n_bad_va}  test: {n_bad_te}  → imputing")

        X_tr = np.where(np.isfinite(X_tr), X_tr, np.nan)
        X_va = np.where(np.isfinite(X_va), X_va, np.nan)
        X_te = np.where(np.isfinite(X_te), X_te, np.nan)

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_va = imp.transform(X_va)
        X_te = imp.transform(X_te)

        print(f"  ✓  [{variant_name}] imputed")
    else:
        print(f"  ✓  [{variant_name}] no NaN or Inf — no imputation needed")

    catch22_72_imputed[variant_name] = {
        "X_train": X_tr, "y_train": y_tr,
        "X_val":   X_va, "y_val":   y_va,
        "X_test":  X_te, "y_test":  y_te,
    }


  ⚠️  [hybrid_72_nsep_8000] non-finite values — train: 143743  val: 36505  test: 71239  → imputing
  ✓  [hybrid_72_nsep_8000] imputed
  ⚠️  [hybrid_72_nsep_4000] non-finite values — train: 70896  val: 36505  test: 71239  → imputing
  ✓  [hybrid_72_nsep_4000] imputed


### Classification

In [34]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1     = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [36]:
from sklearn.svm import LinearSVC

best_model_class = LinearSVC
best_params      = {"C": 1, "class_weight": "balanced", "max_iter": 3000}

DATASET_NAMES = {
    "hybrid_72_nsep_8000" : "hybrid_72_nsep_8000",
    "hybrid_72_nsep_4000" : "hybrid_72_nsep_4000",
}

print(f"{'═'*60}")
print(f"  Classifier : SVM")
print(f"  Model      : LinearSVC  C=1  cw=balanced")
print(f"{'═'*60}")

for norm_key, norm_label in DATASET_NAMES.items():
    if norm_key not in catch22_72_imputed:
        print(f"  [{norm_key}] not found — skipping")
        continue

    d = catch22_72_imputed[norm_key]
    print(f"\n  ── Dataset: {norm_label} ──")

    metrics_list = []
    for run in range(2):
        model   = best_model_class(**best_params)
        metrics = train_and_evaluate(
            model,
            d["X_train"], d["y_train"],
            d["X_test"],  d["y_test"]
        )
        metrics_list.append(metrics)
        print(f"    Run {run+1} — TSS={metrics['tss']:.4f}  F1={metrics['f1']:.4f}  "
              f"Train={metrics['train_time']:.2f}s")

    filepath = os.path.join(RESULTS_DIR, f"svm_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"SVM | {norm_label}")

print(f"\n\n✅  All SVM experiments complete.")
print(f"    Results saved to : {os.path.abspath(RESULTS_DIR)}/")
print(f"    Files : {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

════════════════════════════════════════════════════════════
  Classifier : SVM
  Model      : LinearSVC  C=1  cw=balanced
════════════════════════════════════════════════════════════

  ── Dataset: hybrid_72_nsep_8000 ──


/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    Run 1 — TSS=0.4315  F1=0.0907  Train=51.49s


/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    Run 2 — TSS=0.4315  F1=0.0907  Train=51.16s

───────────────────────────────────────────────────────
  SVM | hybrid_72_nsep_8000
───────────────────────────────────────────────────────
  Run 1: TP=18  TN=3180  FP=345  FN=16
         TSS=0.4315  HSS1=0.0745  HSS2=0.0745  GSS=0.0387
         Recall=0.5294  F1=0.0907  Acc=0.8986
         Train=51.49s  Infer=0.0013s

  Run 2: TP=18  TN=3180  FP=345  FN=16
         TSS=0.4315  HSS1=0.0745  HSS2=0.0745  GSS=0.0387
         Recall=0.5294  F1=0.0907  Acc=0.8986
         Train=51.16s  Infer=0.0017s

  ── Average of 2 runs ──
  tss          : 0.4315
  hss1         : 0.0745
  hss2         : 0.0745
  gss          : 0.0387
  recall       : 0.5294
  f1           : 0.0907
  accuracy     : 0.8986
  train_time   : 51.3254
  infer_time   : 0.0015
───────────────────────────────────────────────────────

  ── Dataset: hybrid_72_nsep_4000 ──


/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    Run 1 — TSS=0.3713  F1=0.0800  Train=22.30s
    Run 2 — TSS=0.3713  F1=0.0800  Train=22.25s

───────────────────────────────────────────────────────
  SVM | hybrid_72_nsep_4000
───────────────────────────────────────────────────────
  Run 1: TP=16  TN=3175  FP=350  FN=18
         TSS=0.3713  HSS1=0.0636  HSS2=0.0636  GSS=0.0329
         Recall=0.4706  F1=0.0800  Acc=0.8966
         Train=22.30s  Infer=0.0021s

  Run 2: TP=16  TN=3175  FP=350  FN=18
         TSS=0.3713  HSS1=0.0636  HSS2=0.0636  GSS=0.0329
         Recall=0.4706  F1=0.0800  Acc=0.8966
         Train=22.25s  Infer=0.0005s

  ── Average of 2 runs ──
  tss          : 0.3713
  hss1         : 0.0636
  hss2         : 0.0636
  gss          : 0.0329
  recall       : 0.4706
  f1           : 0.0800
  accuracy     : 0.8966
  train_time   : 22.2711
  infer_time   : 0.0013
───────────────────────────────────────────────────────


✅  All SVM experiments complete.
    Results saved to : /Users/samskanderi/SEP_DataAugmentation/resu

/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


### GRU 

In [27]:
import numpy as np
from sklearn.metrics import confusion_matrix
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    tss       = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected  = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1      = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom     = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2      = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss         = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }

def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")

def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

In [45]:
# ══════════════════════════════════════════════════════════════
# GRU — PyTorch
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import numpy as np
import time

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model


# ══════════════════════════════════════════════════════════════
# RUN EXPERIMENTS — hybrid_144_nsep_8000 & hybrid_144_nsep_4000
# ══════════════════════════════════════════════════════════════
best_params = {"units": 128, "dropout": 0.3, "layers": 1, "lr": 0.001, "batch_size": 32}

TARGET_VARIANTS = ["hybrid_72_nsep_8000", "hybrid_72_nsep_4000"]

print(f"{'═'*60}")
print(f"  Classifier : GRU (PyTorch)")
print(f"  Best params: {best_params}")
print(f"{'═'*60}")

for variant_name in TARGET_VARIANTS:
    if variant_name not in undersampled_datasets:
        print(f"  [{variant_name}] not found — skipping")
        continue

    d  = undersampled_datasets[variant_name]
    cr = int((d["y_train"] == 0).sum() / max((d["y_train"] == 1).sum(), 1))
    print(f"\n  ── Dataset: {variant_name}  (class ratio {cr}:1) ──")
    print(f"     Train: NSEP={(d['y_train']==0).sum():.0f}  SEP={(d['y_train']==1).sum():.0f}")

    metrics_list = []
    for run in range(2):
        print(f"    Run {run+1} …", flush=True)
        metrics, _ = train_and_evaluate_gru(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio = cr
        )
        metrics_list.append(metrics)
        print(f"    Run {run+1} — TSS={metrics['tss']:.4f}  F1={metrics['f1']:.4f}  "
              f"Train={metrics['train_time']:.1f}s")

    filepath = os.path.join(RESULTS_DIR, f"gru_{variant_name}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"GRU | {variant_name}")

print(f"\n✅  All GRU experiments complete.")
print(f"    Files : {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

Using device: mps
════════════════════════════════════════════════════════════
  Classifier : GRU (PyTorch)
  Best params: {'units': 128, 'dropout': 0.3, 'layers': 1, 'lr': 0.001, 'batch_size': 32}
════════════════════════════════════════════════════════════

  ── Dataset: hybrid_72_nsep_8000  (class ratio 66:1) ──
     Train: NSEP=7829  SEP=118
    Run 1 …
    Run 1 — TSS=0.5823  F1=0.1183  Train=76.5s
    Run 2 …
    Run 2 — TSS=0.6343  F1=0.1205  Train=84.6s

───────────────────────────────────────────────────────
  GRU | hybrid_72_nsep_8000
───────────────────────────────────────────────────────
  Run 1: TP=23  TN=3193  FP=332  FN=11
         TSS=0.5823  HSS1=0.1026  HSS2=0.1026  GSS=0.0541
         Recall=0.6765  F1=0.1183  Acc=0.9036
         Train=76.52s  Infer=0.0889s

  Run 2: TP=25  TN=3169  FP=356  FN=9
         TSS=0.6343  HSS1=0.1048  HSS2=0.1048  GSS=0.0553
         Recall=0.7353  F1=0.1205  Acc=0.8974
         Train=84.59s  Infer=0.0879s

  ── Average of 2 runs ──
  tss 